# Phase 07E: T1-S1 Quantized Validation Quality Gate

Notebook này đánh giá ứng viên INT8 đã được Phase 07D chọn cho **T1-S1**
so với chính FP32 control cùng checkpoint trên toàn bộ Karpathy Validation.

Protocol khóa:

- Student `S1`, seed `123`, checkpoint không thay đổi;
- 5.000 ảnh và 25.010 caption Validation;
- Beam search 3, length penalty 0,7, tối đa 31 bước;
- metric: BLEU-1, BLEU-4, METEOR, ROUGE-L, CIDEr-D và CE;
- gate INT8 − FP32: CIDEr-D ≥ −0,02; BLEU-4 ≥ −0,01; CE ≤ +0,05;
- CPU/QNNPACK; kết quả tốc độ chỉ là tham chiếu, không thay Android vật lý;
- hỗ trợ resume theo image boundary và tự dừng an toàn trước giới hạn Kaggle;
- không huấn luyện, không chọn lại checkpoint và không mở Test.

Ba notebook T1–S1, T2–S2 và T3–S3 dùng cùng code/protocol và có thể chạy
song song. Sau khi cả ba hoàn tất, Phase 07F mới tổng hợp quyết định cuối.

## Kaggle inputs

1. Phase 07D Mobile Runtime Quantization Preflight.
2. Phase 07C Export Parity Validation CPU Benchmark.
3. Phase 07B Portable Model Factory Export.
4. Phase 02A Shared Student Pipeline.
5. Phase 01 MS COCO Data Contract.
6. Dataset `mscoco2014-1`.
7. Offline `pycocoevalcap==1.2`.
8. Khi resume: output 07E trước của đúng cặp `T1-S1`.

## Runtime

- Accelerator: `None / CPU`.
- Internet: `Off`.
- Không gắn Test.


## Cell 1 — Authorization, manifest và pair contract


In [1]:
import csv, hashlib, importlib.util, json, math, os, platform, shutil
import subprocess, sys, time
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

STARTED = time.perf_counter()
PAIR = "T1-S1"
PAIR_SLUG = "t1_s1"
STUDENT_ID = "S1"
SEED = 123
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    f"/kaggle/working//phase7e_{PAIR_SLUG}_quantized_validation_quality",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
PREDICTION = OUT / "predictions"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, PREDICTION, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass
torch.set_grad_enabled(False)
torch.manual_seed(42)
np.random.seed(42)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)

def sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}; tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target == root or root in target.parents
    return target

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"]) == int(marker["manifest_rows"])
    for row in rows:
        artifact = resolve_inside(marker_path.parent, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

P7D_PATH = exactly_one(
    "PHASE7D_MOBILE_RUNTIME_QUANTIZATION_PREFLIGHT_COMPLETE.json"
)
AUTH_PATH = exactly_one(
    "PHASE7E_VALIDATION_QUALITY_OR_FP32_PACKAGING_AUTHORIZED.json"
)
P7C_PATH = exactly_one(
    "PHASE7C_EXPORT_PARITY_VALIDATION_CPU_BENCHMARK_COMPLETE.json"
)
P7B_PATH = exactly_one(
    "PHASE7B_PORTABLE_MODEL_FACTORY_EXPORT_COMPLETE.json"
)
P2_PATH = exactly_one("PHASE2_SHARED_TRAINING_BUNDLE_READY.json")
P1_PATH = exactly_one("PHASE1_FORMAL_DELIVERABLES_READY.json")
P7D, AUTH, P7C, P7B, P2, P1 = map(
    read_json, [P7D_PATH, AUTH_PATH, P7C_PATH, P7B_PATH, P2_PATH, P1_PATH]
)

assert P7D["status"] == "complete"
assert P7D["decision"] == (
    "compatibility_preflight_complete_phase7e_quantized_quality_authorized"
)
assert AUTH["status"] == "authorized"
assert AUTH["mode"] == "quantized_validation_quality_gate"
assert AUTH["new_training"] is False
assert AUTH["checkpoint_reselection"] is False
assert AUTH["test_set_status"] == "sealed" and AUTH["test_set_used"] is False
assert P7C["status"] == "complete" and P7C["fp32_parity"] == "3 / 3 passed"
assert P7B["status"] == "complete"
assert P2["status"] == "ready"
assert P1["status"] == "ready"
assert P1["test_set_status"] == "sealed" and P1["test_used"] is False

p7d_rows = verify_manifest(P7D_PATH, P7D)
selected = [
    row for row in AUTH["selected_quantized_candidates"]
    if row["pair"] == PAIR
]
assert len(selected) == 1, selected
SELECTED = selected[0]
assert SELECTED["student_id"] == STUDENT_ID
assert int(SELECTED["seed"]) == SEED
assert SELECTED["eligible_for_phase7e"] is True

QUALITY_GATES = AUTH["quality_protocol"]["quantized_minus_fp32_gates"]
assert QUALITY_GATES == {
    "CIDEr-D_minimum_delta": -0.02,
    "BLEU-4_minimum_delta": -0.01,
    "CE_maximum_delta": 0.05,
}
write_json(AUDIT / "phase7e_cell01_entry.json", {
    "schema_version": "phase7e_cell01_entry_v1",
    "created_at_utc": now(),
    "status": "passed",
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": SELECTED["strategy"],
    "phase7d_manifest_rows": len(p7d_rows),
    "quality_gates": QUALITY_GATES,
    "runtime": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu_threads": torch.get_num_threads(),
    },
    "new_training_started": False,
    "checkpoint_reselection": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07E CELL 1 — ENTRY")
print("Status                  : PASSED")
print("Pair / seed             :", PAIR, "/", SEED)
print("Strategy                :", SELECTED["strategy"])
print("Phase 07D manifest      :", len(p7d_rows), "/", len(p7d_rows), "VERIFIED")
print("New training            : NOT STARTED")
print("Held-out Test           : SEALED")
print("Progress                : 1 / 6 code cells")


PHASE 07E CELL 1 — ENTRY
Status                  : PASSED
Pair / seed             : T1-S1 / 123
Strategy                : encoder_decoder_dynamic_int8_linear
Phase 07D manifest      : 23 / 23 VERIFIED
New training            : NOT STARTED
Held-out Test           : SEALED
Progress                : 1 / 6 code cells


## Cell 2 — Validation corpus, vocabulary, images và COCO metrics


In [2]:
STARTED = time.perf_counter()
import pyarrow.parquet as pq
from PIL import Image
from torchvision import transforms
from torchvision.transforms import InterpolationMode

P1_ROOT = P1_PATH.parent
P2_ROOT = P2_PATH.parent
VAL_PARQUET = P1_ROOT / "parquet/mscoco_validation.parquet"
VOCAB_PATH = P1_ROOT / "vocabulary/word_vocabulary_minfreq5.json"
assert VAL_PARQUET.is_file() and VOCAB_PATH.is_file()

table = pq.read_table(
    VAL_PARQUET,
    columns=[
        "annotation_id", "image_id", "karpathy_split",
        "source_partition", "caption", "token_ids",
    ],
)
VALIDATION_REFERENCES = defaultdict(list)
VALIDATION_TOKEN_ROWS = defaultdict(list)
for row in table.to_pylist():
    assert row["karpathy_split"] == "val"
    assert row["source_partition"] == "val2014"
    image_id = int(row["image_id"])
    VALIDATION_REFERENCES[image_id].append(str(row["caption"]))
    VALIDATION_TOKEN_ROWS[image_id].append((
        int(row["annotation_id"]),
        tuple(int(value) for value in row["token_ids"]),
    ))
for image_id in VALIDATION_TOKEN_ROWS:
    VALIDATION_TOKEN_ROWS[image_id].sort()
VALIDATION_IMAGE_IDS = tuple(sorted(VALIDATION_REFERENCES))
assert len(VALIDATION_IMAGE_IDS) == 5000
assert sum(map(len, VALIDATION_REFERENCES.values())) == 25010

vocabulary = read_json(VOCAB_PATH)
raw_id_to_token = vocabulary["id_to_token"]
ID_TO_TOKEN = {
    index: token for index, token in enumerate(raw_id_to_token)
}
assert len(ID_TO_TOKEN) == 10080
assert [ID_TO_TOKEN[index] for index in range(4)] == [
    "<pad>", "<bos>", "<eos>", "<unk>",
]

def decode_tokens(token_ids):
    words = []
    for token_id in token_ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id in {0, 1, 3}:
            continue
        words.append(ID_TO_TOKEN[token_id])
    return " ".join(words)

preprocess_path = (
    P2_ROOT
    / f"configs/preprocessing/{STUDENT_ID.lower()}_preprocessing_contract.json"
)
preprocess_contract = read_json(preprocess_path)
validation_geometry = preprocess_contract["shared_geometry"]["validation"]
VALIDATION_TRANSFORM = transforms.Compose([
    transforms.Resize(
        int(validation_geometry["resize_shorter"]),
        interpolation=InterpolationMode.BICUBIC,
        antialias=True,
    ),
    transforms.CenterCrop(int(validation_geometry["center_crop"])),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=preprocess_contract["normalization"]["mean"],
        std=preprocess_contract["normalization"]["std"],
    ),
])

known_val_roots = [
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014",
    INPUT_ROOT / "datasets/nguynthnhdan/mscoco2014-1/val2014/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014",
    INPUT_ROOT / "mscoco2014-1/val2014/val2014",
]
candidate_roots = {
    path.resolve() for path in known_val_roots if path.is_dir()
}
for pattern in ["*/val2014", "*/*/val2014", "*/*/*/val2014", "*/*/*/*/val2014"]:
    for path in INPUT_ROOT.glob(pattern):
        if path.is_dir():
            candidate_roots.add(path.resolve())
            if (path / "val2014").is_dir():
                candidate_roots.add((path / "val2014").resolve())

def filename(image_id):
    return f"COCO_val2014_{int(image_id):012d}.jpg"

probe_ids = VALIDATION_IMAGE_IDS[:16]
valid_roots = [
    root for root in sorted(candidate_roots)
    if all((root / filename(image_id)).is_file() for image_id in probe_ids)
]
if not valid_roots:
    dataset_root = INPUT_ROOT / "datasets"
    search_root = dataset_root if dataset_root.is_dir() else INPUT_ROOT
    recursive = {
        path.resolve() for path in search_root.rglob("val2014") if path.is_dir()
    }
    recursive |= {
        (path / "val2014").resolve()
        for path in recursive if (path / "val2014").is_dir()
    }
    valid_roots = [
        root for root in sorted(recursive)
        if all((root / filename(image_id)).is_file() for image_id in probe_ids)
    ]
assert len(valid_roots) == 1, valid_roots
VAL_ROOT = valid_roots[0]

try:
    from pycocoevalcap.bleu.bleu import Bleu
except Exception:
    wheel_matches = sorted(
        INPUT_ROOT.rglob("pycocoevalcap-1.2-py3-none-any.whl")
    )
    assert len(wheel_matches) == 1, wheel_matches
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "--no-index", str(wheel_matches[0]),
        ],
        check=True,
    )
    from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer

def compute_coco_metrics(references, hypotheses):
    ground_truth = {
        int(image_id): [
            {"caption": caption} for caption in captions
        ]
        for image_id, captions in references.items()
    }
    predictions = {
        int(image_id): [{"caption": hypotheses[int(image_id)]}]
        for image_id in references
    }
    tokenizer = PTBTokenizer()
    tokenized_gt = tokenizer.tokenize(ground_truth)
    tokenized_pred = tokenizer.tokenize(predictions)
    bleu, _ = Bleu(4).compute_score(tokenized_gt, tokenized_pred)
    meteor, _ = Meteor().compute_score(tokenized_gt, tokenized_pred)
    rouge, _ = Rouge().compute_score(tokenized_gt, tokenized_pred)
    cider, _ = Cider().compute_score(tokenized_gt, tokenized_pred)
    return {
        "BLEU-1": float(bleu[0]),
        "BLEU-4": float(bleu[3]),
        "METEOR": float(meteor),
        "ROUGE-L": float(rouge),
        "CIDEr-D": float(cider),
    }

smoke_refs = {
    image_id: VALIDATION_REFERENCES[image_id]
    for image_id in VALIDATION_IMAGE_IDS[:16]
}
smoke_hyps = {
    image_id: VALIDATION_REFERENCES[image_id][0]
    for image_id in VALIDATION_IMAGE_IDS[:16]
}
smoke_metrics = compute_coco_metrics(smoke_refs, smoke_hyps)
assert all(math.isfinite(value) for value in smoke_metrics.values())

write_json(AUDIT / "phase7e_cell02_validation_data.json", {
    "schema_version": "phase7e_cell02_validation_data_v1",
    "created_at_utc": now(),
    "status": "passed",
    "validation_images": len(VALIDATION_IMAGE_IDS),
    "validation_references": sum(map(len, VALIDATION_REFERENCES.values())),
    "vocabulary_size": len(ID_TO_TOKEN),
    "metrics_smoke": smoke_metrics,
    "caption_text_read": True,
    "test_parquet_opened": False,
    "test_images_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07E CELL 2 — VALIDATION DATA")
print("Status                  : PASSED")
print("Validation images       : 5,000")
print("Validation references   : 25,010")
print("Vocabulary              : 10,080")
print("COCO metrics smoke      : PASSED")
print("Held-out Test           : SEALED / NOT READ")
print("Progress                : 2 / 6 code cells")


Processing /kaggle/input/datasets/nguynthnhdan/pycocoevalcap-1-2/pycocoevalcap-1.2-py3-none-any.whl


PTBTokenizer tokenized 997 tokens at 9678.17 tokens per second.
PTBTokenizer tokenized 200 tokens at 2707.22 tokens per second.


{'testlen': 171, 'reflen': 171, 'guess': [171, 155, 139, 123], 'correct': [171, 155, 139, 123]}
ratio: 0.9999999999941521
PHASE 07E CELL 2 — VALIDATION DATA
Status                  : PASSED
Validation images       : 5,000
Validation references   : 25,010
Vocabulary              : 10,080
COCO metrics smoke      : PASSED
Held-out Test           : SEALED / NOT READ
Progress                : 2 / 6 code cells


## Cell 3 — Resolve FP32/INT8 artifacts và runtime smoke


In [3]:
STARTED = time.perf_counter()
P7D_ROOT = P7D_PATH.parent
P7C_ROOT = P7C_PATH.parent
P7B_ROOT = P7B_PATH.parent

torch.backends.quantized.engine = SELECTED["quantized_engine"]
artifact_map = {item["role"]: item for item in SELECTED["artifacts"]}
INT8_ENCODER_PATH = resolve_inside(
    P7D_ROOT, artifact_map["encoder"]["relative_path"]
)
INT8_DECODER_PATH = resolve_inside(
    P7D_ROOT, artifact_map["decoder"]["relative_path"]
)
assert sha256(INT8_ENCODER_PATH) == artifact_map["encoder"]["sha256"]
assert sha256(INT8_DECODER_PATH) == artifact_map["decoder"]["sha256"]

p7b_registry = read_json(P7B_ROOT / "results/phase7b_export_registry.json")
p7b_row = [row for row in p7b_registry["rows"] if row["pair"] == PAIR]
assert len(p7b_row) == 1
p7b_artifacts = {
    item["format"]: item for item in p7b_row[0]["artifacts"]
}
fp32_decoder_info = p7b_artifacts["torchscript_decoder"]
FP32_DECODER_PATH = resolve_inside(
    P7B_ROOT, fp32_decoder_info["relative_path"]
)
assert sha256(FP32_DECODER_PATH) == fp32_decoder_info["sha256"]
if PAIR == "T3-S3":
    repair = read_json(
        P7C_ROOT / "audits/phase7c_t3_encoder_export_repair.json"
    )
    fp32_encoder_info = repair["repaired_artifact"]
    FP32_ENCODER_PATH = resolve_inside(
        P7C_ROOT, fp32_encoder_info["relative_path"]
    )
else:
    fp32_encoder_info = p7b_artifacts["torchscript_encoder"]
    FP32_ENCODER_PATH = resolve_inside(
        P7B_ROOT, fp32_encoder_info["relative_path"]
    )
assert sha256(FP32_ENCODER_PATH) == fp32_encoder_info["sha256"]

FP32_ENCODER = torch.jit.load(
    str(FP32_ENCODER_PATH), map_location="cpu"
).eval()
FP32_DECODER = torch.jit.load(
    str(FP32_DECODER_PATH), map_location="cpu"
).eval()
INT8_ENCODER = torch.jit.load(
    str(INT8_ENCODER_PATH), map_location="cpu"
).eval()
INT8_DECODER = torch.jit.load(
    str(INT8_DECODER_PATH), map_location="cpu"
).eval()

smoke_id = VALIDATION_IMAGE_IDS[0]
with Image.open(VAL_ROOT / filename(smoke_id)) as image_handle:
    smoke_image = VALIDATION_TRANSFORM(
        image_handle.convert("RGB")
    ).unsqueeze(0)
smoke_ids = torch.zeros((3, 31), dtype=torch.long)
smoke_ids[:, 0] = 1
smoke_mask = smoke_ids.ne(0)
with torch.inference_mode():
    fp32_memory = FP32_ENCODER(smoke_image)
    int8_memory = INT8_ENCODER(smoke_image)
BATCHED_DECODER_SUPPORTED = True
BATCHED_DECODER_ERROR = None
try:
    with torch.inference_mode():
        fp32_batch_logits = FP32_DECODER(
            smoke_ids,
            fp32_memory.expand(3, -1, -1).contiguous(),
            smoke_mask,
        )
        int8_batch_logits = INT8_DECODER(
            smoke_ids,
            int8_memory.expand(3, -1, -1).contiguous(),
            smoke_mask,
        )
    assert tuple(fp32_batch_logits.shape) == (3, 31, 10080)
    assert tuple(int8_batch_logits.shape) == (3, 31, 10080)
    assert torch.isfinite(fp32_batch_logits).all()
    assert torch.isfinite(int8_batch_logits).all()
except Exception as error:
    BATCHED_DECODER_SUPPORTED = False
    BATCHED_DECODER_ERROR = f"{type(error).__name__}: {error}"
    with torch.inference_mode():
        fp32_single_logits = FP32_DECODER(
            smoke_ids[:1], fp32_memory, smoke_mask[:1]
        )
        int8_single_logits = INT8_DECODER(
            smoke_ids[:1], int8_memory, smoke_mask[:1]
        )
    assert tuple(fp32_single_logits.shape) == (1, 31, 10080)
    assert tuple(int8_single_logits.shape) == (1, 31, 10080)
    assert torch.isfinite(fp32_single_logits).all()
    assert torch.isfinite(int8_single_logits).all()

write_json(CONFIG / "phase7e_pair_quality_contract.json", {
    "schema_version": "phase7e_pair_quality_contract_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": SELECTED["strategy"],
    "quantized_engine": SELECTED["quantized_engine"],
    "fp32": {
        "encoder_sha256": sha256(FP32_ENCODER_PATH),
        "decoder_sha256": sha256(FP32_DECODER_PATH),
    },
    "int8": {
        "encoder_sha256": sha256(INT8_ENCODER_PATH),
        "decoder_sha256": sha256(INT8_DECODER_PATH),
    },
    "decoding": {
        "algorithm": "beam search",
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_decoder_steps": 31,
        "minimum_lexical_tokens_before_eos": 2,
        "forbidden_token_ids": [0, 1, 3],
    },
    "quality_gates": QUALITY_GATES,
    "batch_decoder_supported": BATCHED_DECODER_SUPPORTED,
    "batch_decoder_error": BATCHED_DECODER_ERROR,
    "test_set_status": "sealed",
    "test_set_used": False,
})
write_json(AUDIT / "phase7e_cell03_runtime_smoke.json", {
    "schema_version": "phase7e_cell03_runtime_smoke_v1",
    "created_at_utc": now(),
    "status": "passed",
    "fp32_artifacts_verified": "2 / 2",
    "int8_artifacts_verified": "2 / 2",
    "batch3_decoder_smoke": (
        "passed" if BATCHED_DECODER_SUPPORTED else "fallback_batch1"
    ),
    "output_shape": (
        [3, 31, 10080]
        if BATCHED_DECODER_SUPPORTED else [1, 31, 10080]
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07E CELL 3 — RUNTIME")
print("Status                  : PASSED")
print("FP32 artifacts          : 2 / 2 VERIFIED")
print("INT8 artifacts          : 2 / 2 VERIFIED")
print(
    "Batch-3 decoder         :",
    "PASSED" if BATCHED_DECODER_SUPPORTED else "BATCH-1 FALLBACK",
)
print("Quantized engine        :", SELECTED["quantized_engine"])
print("Held-out Test           : SEALED")
print("Progress                : 3 / 6 code cells")


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: Currently, qnnpack incorrectly ignores reduce_range when it is set to true; this may change in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/quantized/cpu/qlinear_dynamic.cpp:251.)
  return forward_call(*args, **kwargs)


PHASE 07E CELL 3 — RUNTIME
Status                  : PASSED
FP32 artifacts          : 2 / 2 VERIFIED
INT8 artifacts          : 2 / 2 VERIFIED
Batch-3 decoder         : PASSED
Quantized engine        : qnnpack
Held-out Test           : SEALED
Progress                : 3 / 6 code cells


## Cell 4 — Full Validation generation, CE và exact resume


In [4]:
STARTED = time.perf_counter()
PRED_PATH = PREDICTION / f"{PAIR_SLUG}_paired_predictions.jsonl"
SESSION_BUDGET_SECONDS = float(os.environ.get(
    "KLCN_SESSION_BUDGET_SECONDS", str(10.0 * 3600)
))

def read_jsonl(path):
    rows = {}
    if not Path(path).is_file():
        return rows
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                row = json.loads(line)
                image_id = int(row["image_id"])
                assert image_id not in rows
                rows[image_id] = row
    return rows

def append_jsonl(path, row):
    with Path(path).open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

# Một JSONL giao dịch chứa cả FP32 và INT8 cho từng ảnh. Vì mỗi dòng chỉ
# được append sau khi hai nhánh hoàn tất, resume không thể lệch hai model.
if not PRED_PATH.exists():
    matches = sorted(NOTEBOOK_ROOT.rglob(PRED_PATH.name))
    assert len(matches) <= 1, matches
    if matches:
        shutil.copy2(matches[0], PRED_PATH)

paired_rows = read_jsonl(PRED_PATH)
assert set(paired_rows).issubset(set(VALIDATION_IMAGE_IDS))
fp32_rows = {
    image_id: row["fp32"] for image_id, row in paired_rows.items()
}
int8_rows = {
    image_id: row["int8"] for image_id, row in paired_rows.items()
}

def beam_generate(encoder, decoder, image):
    started_ns = time.perf_counter_ns()
    with torch.inference_mode():
        memory = encoder(image)
        beams = [([1], 0.0, False)]
        for _ in range(31):
            active_indices = [
                index for index, beam in enumerate(beams) if not beam[2]
            ]
            if not active_indices:
                break
            active = [beams[index] for index in active_indices]
            decoder_ids = torch.zeros(
                (len(active), 31), dtype=torch.long
            )
            decoder_mask = torch.zeros(
                (len(active), 31), dtype=torch.bool
            )
            for row_index, (tokens, _, _) in enumerate(active):
                length = min(len(tokens), 31)
                decoder_ids[row_index, :length] = torch.tensor(
                    tokens[:length], dtype=torch.long
                )
                decoder_mask[row_index, :length] = True
            expanded_memory = memory.expand(
                len(active), -1, -1
            ).contiguous()
            if BATCHED_DECODER_SUPPORTED:
                logits = decoder(
                    decoder_ids, expanded_memory, decoder_mask
                )
            else:
                logits = torch.cat([
                    decoder(
                        decoder_ids[index:index + 1],
                        memory,
                        decoder_mask[index:index + 1],
                    )
                    for index in range(len(active))
                ], dim=0)
            candidates = [
                beam for beam in beams if beam[2]
            ]
            for row_index, (tokens, score, _) in enumerate(active):
                position = min(len(tokens) - 1, 30)
                log_probs = F.log_softmax(
                    logits[row_index, position].float(), dim=-1
                )
                log_probs[[0, 1, 3]] = -torch.inf
                lexical_count = max(0, len(tokens) - 1)
                if lexical_count < 2:
                    log_probs[2] = -torch.inf
                values, indices = torch.topk(log_probs, 3)
                for value, token_id in zip(values.tolist(), indices.tolist()):
                    candidates.append((
                        tokens + [int(token_id)],
                        float(score + value),
                        int(token_id) == 2,
                    ))
            def rank(item):
                length = max(1, len(item[0]) - 1)
                return item[1] / (length ** 0.7)
            beams = sorted(candidates, key=rank, reverse=True)[:3]
            if all(item[2] for item in beams):
                break
        best = max(
            beams,
            key=lambda item: item[1] / (
                max(1, len(item[0]) - 1) ** 0.7
            ),
        )
    latency_ms = (time.perf_counter_ns() - started_ns) / 1e6
    return best[0], best[1], memory, latency_ms

def ce_for_image(decoder, memory, token_rows):
    sequences = [tokens for _, tokens in token_rows]
    batch = torch.zeros((len(sequences), 32), dtype=torch.long)
    for index, tokens in enumerate(sequences):
        length = min(len(tokens), 32)
        batch[index, :length] = torch.tensor(
            tokens[:length], dtype=torch.long
        )
    decoder_ids = batch[:, :-1]
    labels = batch[:, 1:]
    mask = decoder_ids.ne(0)
    with torch.inference_mode():
        if BATCHED_DECODER_SUPPORTED:
            logits = decoder(
                decoder_ids,
                memory.expand(
                    len(sequences), -1, -1
                ).contiguous(),
                mask,
            )
            loss_sum = F.cross_entropy(
                logits.float().reshape(-1, 10080),
                labels.reshape(-1),
                ignore_index=0,
                reduction="sum",
            )
        else:
            loss_sum = torch.zeros((), dtype=torch.float32)
            for index in range(len(sequences)):
                logits = decoder(
                    decoder_ids[index:index + 1],
                    memory,
                    mask[index:index + 1],
                )
                loss_sum = loss_sum + F.cross_entropy(
                    logits.float().reshape(-1, 10080),
                    labels[index:index + 1].reshape(-1),
                    ignore_index=0,
                    reduction="sum",
                )
    token_count = int(labels.ne(0).sum())
    return float(loss_sum), token_count

completed = set(paired_rows)
INTERRUPTED_SAFELY = False
run_started = time.perf_counter()
for ordinal, image_id in enumerate(VALIDATION_IMAGE_IDS):
    if image_id in completed:
        continue
    with Image.open(VAL_ROOT / filename(image_id)) as image_handle:
        image = VALIDATION_TRANSFORM(
            image_handle.convert("RGB")
        ).unsqueeze(0)

    fp32_tokens, fp32_score, fp32_memory, fp32_latency = beam_generate(
        FP32_ENCODER, FP32_DECODER, image
    )
    int8_tokens, int8_score, int8_memory, int8_latency = beam_generate(
        INT8_ENCODER, INT8_DECODER, image
    )
    fp32_ce_sum, fp32_ce_tokens = ce_for_image(
        FP32_DECODER, fp32_memory, VALIDATION_TOKEN_ROWS[image_id]
    )
    int8_ce_sum, int8_ce_tokens = ce_for_image(
        INT8_DECODER, int8_memory, VALIDATION_TOKEN_ROWS[image_id]
    )
    assert fp32_ce_tokens == int8_ce_tokens > 0
    fp32_row = {
        "image_id": image_id,
        "ordinal": ordinal,
        "caption": decode_tokens(fp32_tokens),
        "token_ids": fp32_tokens,
        "beam_score": fp32_score,
        "latency_ms": fp32_latency,
        "ce_loss_sum": fp32_ce_sum,
        "ce_tokens": fp32_ce_tokens,
    }
    int8_row = {
        "image_id": image_id,
        "ordinal": ordinal,
        "caption": decode_tokens(int8_tokens),
        "token_ids": int8_tokens,
        "beam_score": int8_score,
        "latency_ms": int8_latency,
        "ce_loss_sum": int8_ce_sum,
        "ce_tokens": int8_ce_tokens,
    }
    paired_row = {
        "image_id": image_id,
        "ordinal": ordinal,
        "fp32": fp32_row,
        "int8": int8_row,
    }
    append_jsonl(PRED_PATH, paired_row)
    paired_rows[image_id] = paired_row
    fp32_rows[image_id] = fp32_row
    int8_rows[image_id] = int8_row
    completed.add(image_id)

    if len(completed) % 50 == 0:
        print(
            f"{PAIR} | completed={len(completed):04d}/5000 | "
            f"fp32={fp32_latency:.1f} ms | int8={int8_latency:.1f} ms"
        )
    if time.perf_counter() - run_started >= SESSION_BUDGET_SECONDS:
        INTERRUPTED_SAFELY = True
        print("Session budget reached at image boundary:", len(completed))
        break

GENERATION_COMPLETE = len(completed) == 5000
write_json(RESULT / "phase7e_run_state.json", {
    "schema_version": "phase7e_pair_run_state_v1",
    "created_at_utc": now(),
    "pair": PAIR,
    "status": "complete" if GENERATION_COMPLETE else "resume_ready",
    "completed_images": len(completed),
    "total_images": 5000,
    "interrupted_safely": INTERRUPTED_SAFELY,
    "paired_prediction_sha256": sha256(PRED_PATH),
    "test_set_status": "sealed",
    "test_set_used": False,
})

print("PHASE 07E CELL 4 — GENERATION")
print("Status                  :", "COMPLETE" if GENERATION_COMPLETE else "RESUME READY")
print("Completed images        :", len(completed), "/ 5000")
print("FP32 / INT8 rows        :", len(fp32_rows), "/", len(int8_rows))
print("Session interrupted     :", INTERRUPTED_SAFELY)
print("Held-out Test           : SEALED")
print("Progress                : 4 / 6 code cells")


T1-S1 | completed=0050/5000 | fp32=293.3 ms | int8=337.1 ms
T1-S1 | completed=0100/5000 | fp32=275.2 ms | int8=318.9 ms
T1-S1 | completed=0150/5000 | fp32=280.8 ms | int8=321.0 ms
T1-S1 | completed=0200/5000 | fp32=284.7 ms | int8=322.2 ms
T1-S1 | completed=0250/5000 | fp32=272.4 ms | int8=308.2 ms
T1-S1 | completed=0300/5000 | fp32=295.8 ms | int8=339.3 ms
T1-S1 | completed=0350/5000 | fp32=302.2 ms | int8=353.0 ms
T1-S1 | completed=0400/5000 | fp32=269.0 ms | int8=301.6 ms
T1-S1 | completed=0450/5000 | fp32=268.1 ms | int8=306.3 ms
T1-S1 | completed=0500/5000 | fp32=398.4 ms | int8=438.9 ms
T1-S1 | completed=0550/5000 | fp32=317.1 ms | int8=361.4 ms
T1-S1 | completed=0600/5000 | fp32=290.4 ms | int8=330.7 ms
T1-S1 | completed=0650/5000 | fp32=270.4 ms | int8=303.3 ms
T1-S1 | completed=0700/5000 | fp32=291.8 ms | int8=326.9 ms
T1-S1 | completed=0750/5000 | fp32=292.8 ms | int8=335.5 ms
T1-S1 | completed=0800/5000 | fp32=234.5 ms | int8=271.7 ms
T1-S1 | completed=0850/5000 | fp32=327.8

## Cell 5 — COCO metrics, CE và quality decision


In [5]:
STARTED = time.perf_counter()
QUALITY_RESULT = None
if GENERATION_COMPLETE:
    assert set(fp32_rows) == set(int8_rows) == set(VALIDATION_IMAGE_IDS)
    fp32_hypotheses = {
        image_id: fp32_rows[image_id]["caption"]
        for image_id in VALIDATION_IMAGE_IDS
    }
    int8_hypotheses = {
        image_id: int8_rows[image_id]["caption"]
        for image_id in VALIDATION_IMAGE_IDS
    }
    fp32_metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, fp32_hypotheses
    )
    int8_metrics = compute_coco_metrics(
        VALIDATION_REFERENCES, int8_hypotheses
    )
    fp32_ce_tokens = sum(row["ce_tokens"] for row in fp32_rows.values())
    int8_ce_tokens = sum(row["ce_tokens"] for row in int8_rows.values())
    assert fp32_ce_tokens == int8_ce_tokens
    fp32_ce = (
        sum(row["ce_loss_sum"] for row in fp32_rows.values())
        / fp32_ce_tokens
    )
    int8_ce = (
        sum(row["ce_loss_sum"] for row in int8_rows.values())
        / int8_ce_tokens
    )
    delta = {
        key: int8_metrics[key] - fp32_metrics[key]
        for key in fp32_metrics
    }
    delta["CE"] = int8_ce - fp32_ce
    gates = {
        "CIDEr-D": delta["CIDEr-D"] >= QUALITY_GATES["CIDEr-D_minimum_delta"],
        "BLEU-4": delta["BLEU-4"] >= QUALITY_GATES["BLEU-4_minimum_delta"],
        "CE": delta["CE"] <= QUALITY_GATES["CE_maximum_delta"],
        "finite": all(
            math.isfinite(value)
            for value in [
                *fp32_metrics.values(), *int8_metrics.values(),
                fp32_ce, int8_ce,
            ]
        ),
    }
    exact_caption_fraction = sum(
        fp32_hypotheses[image_id] == int8_hypotheses[image_id]
        for image_id in VALIDATION_IMAGE_IDS
    ) / 5000
    quality_passed = all(gates.values())
    QUALITY_RESULT = {
        "schema_version": "phase7e_pair_quality_result_v1",
        "created_at_utc": now(),
        "pair": PAIR,
        "student_id": STUDENT_ID,
        "seed": SEED,
        "strategy": SELECTED["strategy"],
        "selection_split": "Karpathy Validation",
        "validation_images": 5000,
        "validation_references": 25010,
        "fp32": {**fp32_metrics, "CE": fp32_ce},
        "int8": {**int8_metrics, "CE": int8_ce},
        "delta_int8_minus_fp32": delta,
        "gates": gates,
        "quality_gate_passed": quality_passed,
        "deployment_decision": (
            "retain_int8_candidate_for_phase7f"
            if quality_passed else "reject_int8_retain_fp32"
        ),
        "exact_caption_agreement_fraction": exact_caption_fraction,
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(RESULT / "phase7e_quality_result.json", QUALITY_RESULT)
    print("FP32 metrics            :", QUALITY_RESULT["fp32"])
    print("INT8 metrics            :", QUALITY_RESULT["int8"])
    print("Delta INT8-FP32         :", delta)
    print("Quality gate            :", "PASSED" if quality_passed else "FAILED")

write_json(AUDIT / "phase7e_cell05_quality.json", {
    "schema_version": "phase7e_cell05_quality_v1",
    "created_at_utc": now(),
    "status": "passed" if GENERATION_COMPLETE else "deferred_resume_required",
    "generation_complete": GENERATION_COMPLETE,
    "quality_result_available": QUALITY_RESULT is not None,
    "quality_gate_passed": (
        QUALITY_RESULT["quality_gate_passed"]
        if QUALITY_RESULT is not None else None
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})

print("PHASE 07E CELL 5 — QUALITY")
print("Status                  :", "EVALUATED" if QUALITY_RESULT else "DEFERRED")
print("Selection split         : VALIDATION")
print("Held-out Test           : SEALED")
print("Progress                : 5 / 6 code cells")


PTBTokenizer tokenized 307821 tokens at 562709.03 tokens per second.
PTBTokenizer tokenized 52317 tokens at 243240.04 tokens per second.


{'testlen': 47318, 'reflen': 47347, 'guess': [47318, 42318, 37318, 32318], 'correct': [36050, 20296, 10057, 4799]}
ratio: 0.9993875007920038


PTBTokenizer tokenized 307821 tokens at 498139.51 tokens per second.
PTBTokenizer tokenized 52325 tokens at 232801.65 tokens per second.


{'testlen': 47326, 'reflen': 47346, 'guess': [47326, 42326, 37326, 32326], 'correct': [36016, 20248, 10014, 4778]}
ratio: 0.9995775778312636
FP32 metrics            : {'BLEU-1': 0.7613997345618297, 'BLEU-4': 0.34752733899882865, 'METEOR': 0.2667200643710961, 'ROUGE-L': 0.5557211611142506, 'CIDEr-D': 1.1115108313562587, 'CE': 2.3540564535987625}
INT8 metrics            : {'BLEU-1': 0.7606977735147972, 'BLEU-4': 0.346482808166236, 'METEOR': 0.2663071295502023, 'ROUGE-L': 0.5552135825100122, 'CIDEr-D': 1.1096043598291707, 'CE': 2.3541141534469348}
Delta INT8-FP32         : {'BLEU-1': -0.0007019610470325732, 'BLEU-4': -0.0010445308325926494, 'METEOR': -0.0004129348208937733, 'ROUGE-L': -0.0005075786042384456, 'CIDEr-D': -0.001906471527088005, 'CE': 5.76998481722768e-05}
Quality gate            : PASSED
PHASE 07E CELL 5 — QUALITY
Status                  : EVALUATED
Selection split         : VALIDATION
Held-out Test           : SEALED
Progress                : 5 / 6 code cells


## Cell 6 — Package complete/resume state


In [6]:
STARTED = time.perf_counter()
if GENERATION_COMPLETE:
    assert QUALITY_RESULT is not None
    aggregate_ready = {
        "schema_version": "phase7f_pair_input_ready_v1",
        "created_at_utc": now(),
        "status": "ready",
        "pair": PAIR,
        "quality_gate_passed": QUALITY_RESULT["quality_gate_passed"],
        "deployment_decision": QUALITY_RESULT["deployment_decision"],
        "quality_result": {
            "relative_path": "results/phase7e_quality_result.json",
            "sha256": sha256(RESULT / "phase7e_quality_result.json"),
        },
        "test_set_status": "sealed",
        "test_set_used": False,
    }
    write_json(
        OUT / "PHASE7F_QUANTIZED_QUALITY_AGGREGATE_INPUT_READY_T1_S1.json",
        aggregate_ready,
    )

manifest_rows = []
for directory in [AUDIT, CONFIG, RESULT, PREDICTION]:
    for path in sorted(directory.rglob("*")):
        if path.is_file():
            manifest_rows.append({
                "relative_path": path.relative_to(OUT).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256(path),
            })
manifest_csv = MANIFEST / f"phase7e_{PAIR_SLUG}_manifest.csv"
with manifest_csv.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
manifest_json = MANIFEST / f"phase7e_{PAIR_SLUG}_manifest.json"
write_json(manifest_json, {
    "schema_version": "phase7e_pair_manifest_v1",
    "created_at_utc": now(),
    "rows": manifest_rows,
})
sha_file = MANIFEST / "SHA256SUMS"
sha_file.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)

marker = {
    "schema_version": (
        "phase7e_pair_quality_complete_v1"
        if GENERATION_COMPLETE else "phase7e_pair_quality_resume_ready_v1"
    ),
    "created_at_utc": now(),
    "status": "complete" if GENERATION_COMPLETE else "resume_ready",
    "pair": PAIR,
    "student_id": STUDENT_ID,
    "seed": SEED,
    "strategy": SELECTED["strategy"],
    "completed_images": len(completed),
    "validation_images": 5000,
    "quality_gate_passed": (
        QUALITY_RESULT["quality_gate_passed"]
        if QUALITY_RESULT is not None else None
    ),
    "deployment_decision": (
        QUALITY_RESULT["deployment_decision"]
        if QUALITY_RESULT is not None else "resume_required"
    ),
    "new_training_performed": False,
    "checkpoint_reselection_performed": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "manifest": {
        "csv": {
            "relative_path": manifest_csv.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_csv),
            "rows": len(manifest_rows),
        },
        "json": {
            "relative_path": manifest_json.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_json),
        },
        "sha256sums": {
            "relative_path": sha_file.relative_to(OUT).as_posix(),
            "sha256": sha256(sha_file),
        },
    },
    "manifest_rows": len(manifest_rows),
    "elapsed_seconds_cell06": time.perf_counter() - STARTED,
}
marker_name = (
    "PHASE7E_T1_S1_QUANTIZED_VALIDATION_QUALITY_COMPLETE.json"
    if GENERATION_COMPLETE
    else "PHASE7E_T1_S1_QUANTIZED_VALIDATION_QUALITY_RESUME_READY.json"
)
write_json(OUT / marker_name, marker)

print("PHASE 07E — FINAL")
print("Status                  :", marker["status"].upper())
print("Pair                    :", PAIR)
print("Completed images        :", len(completed), "/ 5000")
print("Quality gate            :", marker["quality_gate_passed"])
print("Deployment decision     :", marker["deployment_decision"])
print("New training            : NOT PERFORMED")
print("Held-out Test           : SEALED / NOT USED")
print("Manifest                :", len(manifest_rows), "/", len(manifest_rows), "VERIFIED")
print("Next                    :", "Phase 07F aggregate" if GENERATION_COMPLETE else "attach output and resume this pair")
print("Progress                : 6 / 6 code cells")


PHASE 07E — FINAL
Status                  : COMPLETE
Pair                    : T1-S1
Completed images        : 5000 / 5000
Quality gate            : True
Deployment decision     : retain_int8_candidate_for_phase7f
New training            : NOT PERFORMED
Held-out Test           : SEALED / NOT USED
Manifest                : 8 / 8 VERIFIED
Next                    : Phase 07F aggregate
Progress                : 6 / 6 code cells
